In [ ]:
# -*- coding: utf-8 -*-
"""
K-fold LSTM audio-emotion model  (CMU-MOSEI, 193 handcrafted acoustic features)
Regenerates Table 3 (classification report) and Fig. 7 (confusion matrix)
for the IJST revision.

Run in Google Colab:  Runtime -> Run all.   Upload features_label_12_07_2022.csv
when prompted (or mount Drive and set CSV_PATH).
"""


## [1] Setup --------------------------------------------------------------

In [ ]:
# !pip -q install imbalanced-learn            # uncomment in Colab if needed
import os, sys, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from imblearn.over_sampling import SMOTE

SEED = 0
np.random.seed(SEED); tf.random.set_seed(SEED)

CSV_PATH = "features_label_12_07_2022.csv"
if not os.path.exists(CSV_PATH):
    try:                                   # Colab upload dialog
        from google.colab import files
        CSV_PATH = list(files.upload().keys())[0]
    except ImportError:
        sys.exit(f"CSV not found: {CSV_PATH}")

CLASSES = ["happiness", "sadness", "Angry", "fear", "disgust"]   # label 0..4
FOLDS, EPOCHS_PER_FOLD, FINAL_EPOCHS, BATCH = 5, 20, 40, 32
SMOTE_BEFORE_SPLIT = True   # True = protocol used in the submitted paper; False = leak-free (split first, SMOTE on train only)
OUT_DIR = "outputs"; os.makedirs(OUT_DIR, exist_ok=True)

## [2] Load data -----------------------------------------------------------

In [ ]:
db = pd.read_csv(CSV_PATH)
X1 = db.iloc[:, :-1].values.astype("float32")          # (1395, 193)
y1 = db.iloc[:, -1].values.astype(int)                 # 0..4
print("raw:", X1.shape, dict(zip(*np.unique(y1, return_counts=True))))

## [3]+[4] SMOTE and held-out test split ------------------------------------

In [ ]:
if SMOTE_BEFORE_SPLIT:
    # protocol of the submitted paper: SMOTE on the full set (5 x 826 = 4130), then 20 % test (826)
    X_res, y_res = SMOTE(random_state=SEED).fit_resample(X1, y1)
    src_index = np.r_[np.arange(len(X1)), np.full(len(X_res) - len(X1), -1)]   # -1 = synthetic row
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_res, y_res, src_index, test_size=0.20, random_state=SEED)
else:
    # leak-free protocol: split the 1395 real utterances first, oversample the training part only
    idx_all = np.arange(len(X1))
    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X1, y1, idx_all, test_size=0.20, random_state=SEED, stratify=y1)
    X_train, y_train = SMOTE(random_state=SEED).fit_resample(X_train, y_train)
    idx_train = np.r_[idx_train, np.full(len(X_train) - len(idx_train), -1)]
print("train:", X_train.shape, " test:", X_test.shape, dict(zip(*np.unique(y_test, return_counts=True))))

# save the split for Supplementary Table S1 (original CSV row index; -1 = synthetic SMOTE sample)
pd.DataFrame({"csv_row": idx_train, "label": y_train, "split": "train"}).to_csv(f"{OUT_DIR}/split_train.csv", index=False)
pd.DataFrame({"csv_row": idx_test,  "label": y_test,  "split": "test" }).to_csv(f"{OUT_DIR}/split_test.csv",  index=False)

# feature standardisation (fit on TRAIN only) -- essential: raw features range about -560 .. +460
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train).astype("float32")
X_test  = scaler.transform(X_test).astype("float32")

# The 193 handcrafted features are fed to the LSTM as ONE time-step of 193 channels, (n, 1, 193).
# Feeding them as 193 scalar time-steps, (n, 193, 1), is very slow to train and stays near chance.
x_train3 = X_train.reshape(-1, 1, 193)
x_test3  = X_test.reshape(-1, 1, 193)

## [5] LSTM model (as in the paper: LSTM-512, dropout 0.2, dense 512/256/256/128/128/64/64, softmax-5, RMSprop)

In [ ]:
def build_lstm():
    m = keras.Sequential([
        keras.layers.Input(shape=(1, 193)),
        keras.layers.LSTM(512, return_sequences=False),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(512, activation="relu"), keras.layers.BatchNormalization(),
        keras.layers.Dense(256, activation="relu"), keras.layers.Dropout(0.2),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dense(64,  activation="relu"), keras.layers.Dropout(0.2),
        keras.layers.Dense(5,   activation="softmax"),
    ])
    m.compile(loss="sparse_categorical_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=1e-3), metrics=["accuracy"])
    return m

CALLBACKS = [keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5, verbose=1),
             keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)]

build_lstm().summary()

## [6] K-fold cross-validation on the TRAINING set only (fresh model per fold) ----

In [ ]:
kf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fold_val_acc, histories = [], []
for k, (tr, va) in enumerate(kf.split(X_train, y_train), 1):
    print(f"\n===== Fold {k}/{FOLDS} =====")
    m = build_lstm()
    h = m.fit(x_train3[tr], y_train[tr], validation_data=(x_train3[va], y_train[va]),
              epochs=EPOCHS_PER_FOLD, batch_size=BATCH, verbose=2, callbacks=CALLBACKS)
    histories.append(h.history)
    fold_val_acc.append(max(h.history["val_accuracy"]))
    print(f"fold {k} best validation accuracy: {fold_val_acc[-1]:.4f}")
print("\nK-fold validation accuracy: mean = %.4f +/- %.4f  %s" % (np.mean(fold_val_acc), np.std(fold_val_acc), np.round(fold_val_acc, 4)))

# final model: trained on the whole training set, evaluated ONCE on the held-out test set
print("\n===== Final model on full training set =====")
model = build_lstm()
hist = model.fit(x_train3, y_train, validation_split=0.1, epochs=FINAL_EPOCHS, batch_size=BATCH,
                 verbose=2, callbacks=CALLBACKS)
histories.append(hist.history)
model.save(f"{OUT_DIR}/kfold_lstm_audio_emotion.keras")

## [7] Loss / accuracy curves (concatenated over folds) ----------------------

In [ ]:
loss = sum([h["loss"] for h in histories], []); vloss = sum([h["val_loss"] for h in histories], [])
acc  = sum([h["accuracy"] for h in histories], []); vacc = sum([h["val_accuracy"] for h in histories], [])
for a, b, name in [(loss, vloss, "loss"), (acc, vacc, "accuracy")]:
    plt.figure(); plt.plot(a); plt.plot(b); plt.title(f"model {name}")
    plt.ylabel(name); plt.xlabel("epoch"); plt.legend(["train", "validation"], loc="upper left")
    plt.savefig(f"{OUT_DIR}/lstm_{name}.png", dpi=200, bbox_inches="tight"); plt.show()

## [8] Held-out test evaluation -> Table 3 ------------------------------------

In [ ]:
y_pred = model.predict(x_test3, verbose=0).argmax(axis=1)
print("\nHeld-out test accuracy: %.4f  (%d / %d)" % (accuracy_score(y_test, y_pred),
      (y_test == y_pred).sum(), len(y_test)))
report = classification_report(y_test, y_pred, target_names=CLASSES, digits=2)
print(report)
with open(f"{OUT_DIR}/table3_classification_report.txt", "w") as f:
    f.write(report)

## [9] Confusion matrix -> Fig. 7 ---------------------------------------------

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print(cm)
fig, ax = plt.subplots(figsize=(4.6, 4.0), dpi=200)
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.set_title("Confusion matrix", fontsize=10)
cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04); cb.ax.tick_params(labelsize=7)
ax.set_xticks(range(5)); ax.set_yticks(range(5))
ax.set_xticklabels(CLASSES, rotation=90, fontsize=8); ax.set_yticklabels(CLASSES, fontsize=8)
thresh = cm.max() / 2.0
for i in range(5):
    for j in range(5):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8,
                color="white" if cm[i, j] > thresh else "black")
ax.set_ylabel("True label", fontsize=9); ax.set_xlabel("Predicted label", fontsize=9)
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/fig7_lstm_audio_emotion_cm.png")
plt.show()
np.savetxt(f"{OUT_DIR}/fig7_confusion_matrix.csv", cm, fmt="%d", delimiter=",")
print("\nSaved to", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

## [10] Colab: download everything --------------------------------------------

In [ ]:
try:
    from google.colab import files
    import shutil; shutil.make_archive("lstm_audio_emotion_outputs", "zip", OUT_DIR)
    files.download("lstm_audio_emotion_outputs.zip")
except ImportError:
    pass